# 02 · Create reaches

Re-splits each river's SWORD reaches into RivMapper reaches whose length
scales with drainage area
(`reach_len = length_modulator × len_slope × facc + len_intercept`),
then buffers them and trims the buffers with cut lines perpendicular to the
smoothed valley direction. Outputs go to
`{data_root}/{river}/reaches/`.

Parameters are in the `reaches:` section of `rivmapper.yaml`; any of them
can be overridden per river with a column of the same name in the rivers
table. The logic lives in `rivmapper_v2/reaches.py`. Same thing from a
terminal:

```
rivmapper run projects/oktedi/rivmapper.yaml --steps reaches
```

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt

from rivmapper_v2 import Project, run_pipeline, setup_logging

setup_logging()

# ── Settings ─────────────────────────────────────────────────────────
# Everything else lives in rivmapper.yaml and the rivers table it points to.
CONFIG_PATH = Path("../projects/oktedi/rivmapper.yaml")  # relative to notebooks/
RIVERS = None  # None = all rivers, or e.g. ["Beni", "Bermejo"]
OVERWRITE = False

In [ ]:
project = Project.load(CONFIG_PATH)
if OVERWRITE:
    project.rivers = [r.model_copy(update={"overwrite": True}) for r in project.rivers]
print(project)
print("Rivers:", ", ".join(project.river_names))

## Run

In [ ]:
results = run_pipeline(project, steps=["reaches"], rivers=RIVERS)
results

## QC: reach polygons

Check that cuts are perpendicular to the valley, neighbours do not overlap,
and the head and mouth are squared off. The run record
(`{river}_reaches_run.yaml`) lists the settings used plus diagnostics:
`adjacent_overlap_km2` > 0 means cut lines crossed on a sharp bend (try a
larger `smoothing_window_multiplier`), and `unclipped_reaches` lists
polygons whose cut did not fully cross the buffer.

In [ ]:
import yaml

for river in project.select(RIVERS):
    p = river.paths
    if not p.reach_polygons.exists():
        continue
    polys = gpd.read_file(p.reach_polygons)
    lines = gpd.read_file(p.reach_lines)
    record = yaml.safe_load(p.run_record("reaches").read_text()) if p.run_record("reaches").exists() else {}
    diag = record.get("summary", {})

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    polys.plot(ax=ax, column="ds_order", cmap="tab20", alpha=0.45, edgecolor="k", linewidth=0.4)
    lines.plot(ax=ax, color="navy", linewidth=0.8)
    for _, r in polys.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(str(r.ds_order), (c.x, c.y), fontsize=6, ha="center")
    ax.set_title(
        f"{river.name}: {len(polys)} reaches, "
        f"overlap {diag.get('adjacent_overlap_km2', '?')} km²"
    )
    ax.set_axis_off()
    plt.show()
    display(polys.drop(columns="geometry").round(1))